# **1. 서울 열린데이터 광장

1. "서울시 공공자전거 실시간 대여정보"를 검색합니다.
2. 인증키를 발급 받습니다. 
서울 열린데이터 광장(Seoul Open Data Plaza)은 서울시에서 운영하는 공공데이터 개방 플랫폼입니다. 시민, 연구자, 기업 등이 서울시에서 생성한 다양한 공공데이터를 자유롭게 활용할 수 있도록 제공하고 있습니다. 이를 통해 데이터 기반의 창의적인 아이디어와 혁신을 촉진하며, 시민들의 정보 접근성을 높이고 공공서비스를 개선하는 데 기여하고 있습니다.

In [13]:
import requests
import pandas as pd
import folium

In [2]:
base_url = "http://openapi.seoul.go.kr:8088/65666a6c4a6d736b3532624a664344/json/bikeList/1/5/"
response = requests.get(base_url)
response

<Response [200]>

In [7]:
json_data = response.json()
json_data

{'rentBikeStatus': {'list_total_count': 5,
  'RESULT': {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다.'},
  'row': [{'rackTotCnt': '15',
    'stationName': '102. 망원역 1번출구 앞',
    'parkingBikeTotCnt': '4',
    'shared': '27',
    'stationLatitude': '37.55564880',
    'stationLongitude': '126.91062927',
    'stationId': 'ST-4'},
   {'rackTotCnt': '14',
    'stationName': '103. 망원역 2번출구 앞',
    'parkingBikeTotCnt': '5',
    'shared': '36',
    'stationLatitude': '37.55495071',
    'stationLongitude': '126.91083527',
    'stationId': 'ST-5'},
   {'rackTotCnt': '13',
    'stationName': '104. 합정역 1번출구 앞',
    'parkingBikeTotCnt': '9',
    'shared': '69',
    'stationLatitude': '37.55073929',
    'stationLongitude': '126.91508484',
    'stationId': 'ST-6'},
   {'rackTotCnt': '5',
    'stationName': '105. 합정역 5번출구 앞',
    'parkingBikeTotCnt': '2',
    'shared': '40',
    'stationLatitude': '37.55000687',
    'stationLongitude': '126.91482544',
    'stationId': 'ST-7'},
   {'rackTotCnt': '12',
   

In [6]:
json_data['rentBikeStatus']['RESULT']['CODE']

'INFO-000'

In [8]:
json_data.get('rentBikeStatus', {}).get("RESULT", {}).get("CODE", {})

'INFO-000'

In [9]:
def fetch_bike_data():
    base_url = "http://openapi.seoul.go.kr:8088/65666a6c4a6d736b3532624a664344/json/bikeList/"
    start = 1
    end = 1000
    step = 1000
    data_frames = []

    while True:
        url = f'{base_url}{start}/{end}/'
        response = requests.get(url)
        if response.status_code != 200:
            print(f'status code: {response.status_code}')
            break
        json_data = response.json()
        try:
            rent_bike_status = json_data['rentBikeStatus']
            result_code = rent_bike_status['RESULT']['CODE']
        except KeyError:
            print('JSON 오류')
            break

        if result_code == 'INFO-200':
            print('데이터 없음')
            break
        elif result_code == "INFO-000":
            print(f'시작: {start} 끝: {end}')
            try:
                bike_data = rent_bike_status['row']
                if bike_data:
                    df = pd.DataFrame(bike_data)
                    data_frames.append(df)
            except KeyError:
                print('데이터를 찾을 수 없음')
        else:
            print(f'result code: {result_code}')
            break
        start += step
        end += step

    if data_frames:
        final_df = pd.concat(data_frames, ignore_index=True)
        return final_df
    else:
        return pd.DataFrame()

In [12]:
bike_data_df = fetch_bike_data()
bike_data_df

시작: 1 끝: 1000
시작: 1001 끝: 2000
시작: 2001 끝: 3000
JSON 오류


,rackTotCnt,stationName,parkingBikeTotCnt,shared,stationLatitude,stationLongitude,stationId
0,15,102. 망원역 1번출구 앞,4,27,37.55564880,126.91062927,ST-4
1,14,103. 망원역 2번출구 앞,5,36,37.55495071,126.91083527,ST-5
2,13,104. 합정역 1번출구 앞,8,62,37.55073929,126.91508484,ST-6
3,5,105. 합정역 5번출구 앞,1,20,37.55000687,126.91482544,ST-7
4,12,106. 합정역 7번출구 앞,8,67,37.54864502,126.91282654,ST-8
...,...,...,...,...,...,...,...
2739,12,6187.마곡119안전센터 맞은편,18,150,37.55534744,126.82072449,ST-3415
2740,17,6188.금호아파트,26,153,37.55611038,126.86475372,ST-3419
2741,11,6189.데시앙플렉스 지식산업센터,20,182,37.56448364,126.84830475,ST-3424
2742,10,6190.마곡광장(마곡나루역 6번출구),4,40,37.56614685,126.82738495,ST-3421


In [14]:
bike_data_df['stationLatitude'] = bike_data_df['stationLatitude'].astype(float)
bike_data_df['stationLongitude'] = bike_data_df['stationLongitude'].astype(float)

bike_map = folium.Map(location=[bike_data_df['stationLatitude'].mean(),
                                bike_data_df['stationLongitude'].mean()],
                                zoom_start=12)

for index, data in bike_data_df.iterrows():
    popup_str = '{} 자전거 주차 총 건수:{}대'.format(
        data['stationName'], data['parkingBikeTotCnt']
    )
    popup = folium.Popup(popup_str, max_width=600)
    folium.Marker(location=[data['stationLatitude'], data['stationLongitude']],
                    popup=popup).add_to(bike_map)

bike_map